In [ ]:

import torch
import os, random
import numpy as np
from transformers import set_seed

SEED = 159753

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

set_seed(SEED)

In [ ]:
import pandas as pd


df = pd.read_csv("/run/media/victor/pessoal/mestrado/codigo/database_scripts/V2_leio_with_label.csv")

In [ ]:
df = df[df["is_histopathology"]].reset_index(drop=True).copy()

In [ ]:
df

In [ ]:
def clean_string(string):
    cleaned_string = "".join(char if char.isprintable() else ' ' for char in string)
    return cleaned_string.strip().replace("\n", " ").replace("×", "x").replace("  ", " ").replace("‑", "-").replace("‐","-")


df["caption"] = df["caption"].apply(clean_string)

In [ ]:
df.to_csv("V2_leiomyoma_clean.csv", index=False)

In [ ]:
import spacy
from spacy.language import Language

from spacy_language_detection import LanguageDetector


def get_lang_detector(nlp, name):
    return LanguageDetector(seed=SEED) 


nlp_model = spacy.load("en_core_web_trf", enable=["language_detector", "sentencizer"])
Language.factory("language_detector", func=get_lang_detector)
nlp_model.add_pipe('sentencizer')
nlp_model.add_pipe('language_detector')


In [ ]:
from tqdm import tqdm

CONFIDENCE_THRESHOLD = 0.8

for index, doc in tqdm(enumerate(nlp_model.pipe(df["caption"].values)), total=len(df)):

    language = doc._.language
    score = language["score"]
    lang = language["language"]

    if score >= CONFIDENCE_THRESHOLD:
        df.at[index, "language"] = lang
    else:
        df.at[index, "language"] = "unknown"


In [ ]:
score

In [ ]:
df

In [ ]:
filtered = df[(df["language"] == "en") | (df["language"] == "unknown")].copy()

In [ ]:
filtered

In [ ]:
filtered["new_id"] = filtered["article_id"] + "-" + filtered["id"] + ".jpg"

In [ ]:
filtered

In [ ]:
filtered.to_csv("V2_leiomyoma_language_clean.csv", index=False)

In [ ]:
filtered

In [ ]:
"LMS‐OGC".replace("‐", "-")

In [ ]:
'LMS-OGC'

In [ ]:
   
import shutil

dst_path = "/run/media/victor/pessoal/mestrado/codigo/datasets/V2_leiomyoma_language_clean/"

for index, row in df.iterrows():
    shutil.copy(row["image_path"], dst_path + row["article_id"] + "-" + row["image_path"].split("/")[-1])   
